In [1]:
import pandas as pd
import os
import re
import json

In [3]:


file_path = "../data/Scenario_6"
# pattern to match TCPdump log entries
tcp_pattern = re.compile(
    r'(?P<timestamp>\d+\.\d+).*?\n\s*(?P<src_ip>\d+\.\d+\.\d+\.\d+)\.(?P<src_port>\d+) > '
    r'(?P<dst_ip>\d+\.\d+\.\d+\.\d+)\.(?P<dst_port>\d+): (?P<protocol>[A-Z]+), length (?P<length>\d+)'
)

all_records = []

print(f"Starting processing in: {os.path.abspath(file_path)}")

for video_id in os.listdir(file_path):
    video_path = os.path.join(file_path, video_id)
    if not os.path.isdir(video_path):
        continue

    for iteration in os.listdir(video_path):
        iteration_path = os.path.join(video_path, iteration)
        if not os.path.isdir(iteration_path):
            continue
            
        bandwidth_label = None
        
        try:
            stats_file_name = [f for f in os.listdir(iteration_path) if "Phone_Statistics" in f][0]
            stats_file_path = os.path.join(iteration_path, stats_file_name)

            with open(stats_file_path, 'r') as f:
                line = f.readline()
                json_part = line.split('data=')[1]
                json_string = json_part.strip().strip('"')
                
                stats_data = json.loads(json_string)
                
                if 'bwe' in stats_data:
                    bandwidth_label = stats_data['bwe']
                
        except (IndexError, FileNotFoundError):
            print(f"Warning: No 'Phone_Statistics' file found in {iteration_path}. Skipping.")
            continue
        except (json.JSONDecodeError, KeyError) as e:
            print(f"Warning: Could not parse 'bwe' from {stats_file_path}. Error: {e}. Skipping.")
            continue

        if bandwidth_label is None:
            print(f"Warning: 'bwe' label not found in {stats_file_path}. Skipping.")
            continue

        # Find the TCPdump file and parse it, 
        # also applying the label
        try:
            tcpdump_file_name = [f for f in os.listdir(iteration_path) if "Phone_TCPdump" in f][0]
            tcpdump_file_path = os.path.join(iteration_path, tcpdump_file_name)
            
            with open(tcpdump_file_path, "r") as f:
                content = f.read()
                matches = tcp_pattern.finditer(content)
                for match in matches:
                    record = match.groupdict()
                    record["video_id"] = video_id
                    record["iteration"] = iteration
                    record["bw"] = bandwidth_label
                    all_records.append(record)

        except (IndexError, FileNotFoundError):
            print(f"Warning: No 'Phone_TCPdump' file found in {iteration_path}. Skipping TCPdump for this iteration.")
            continue


if not all_records:
    print("No records were processed. The final DataFrame is empty.")
else:
    df = pd.DataFrame(all_records)

    df["timestamp"] = df["timestamp"].astype(float)
    df["src_port"] = df["src_port"].astype(int)
    df["dst_port"] = df["dst_port"].astype(int)
    df["length"] = df["length"].astype(int)
    df["bw"] = df["bw"].astype(int)

    df = df[['timestamp', 'video_id', 'iteration', 'src_ip', 'src_port', 'dst_ip', 'dst_port', 'protocol', 'length', 'bw']]
    
    #output_filename = 'scenario_6_parsed_logs_with_labels.csv'
    output_filename = '../data/scenario_6_parsed_logs_with_labels.csv'
    df.to_csv(output_filename, index=False)

    print(f"\nProcessing complete. Data saved to '{output_filename}'")
    print("\nDataFrame Info:")
    df.info()
    print("\nDataFrame Head:")
    print(df.head())

Starting processing in: c:\Users\Administrator\Desktop\thesis\data\Scenario_6

Processing complete. Data saved to '../data/scenario_6_parsed_logs_with_labels.csv'

DataFrame Info:
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 6495636 entries, 0 to 6495635
Data columns (total 10 columns):
 #   Column     Dtype  
---  ------     -----  
 0   timestamp  float64
 1   video_id   object 
 2   iteration  object 
 3   src_ip     object 
 4   src_port   int64  
 5   dst_ip     object 
 6   dst_port   int64  
 7   protocol   object 
 8   length     int64  
 9   bw         int64  
dtypes: float64(1), int64(4), object(5)
memory usage: 495.6+ MB

DataFrame Head:
      timestamp         video_id    iteration         src_ip  src_port  \
0  1.517452e+09  Vid_2d1VrCvdzbY  Iteration_1     10.10.0.11     48645   
1  1.517452e+09  Vid_2d1VrCvdzbY  Iteration_1     10.10.0.11     48645   
2  1.517452e+09  Vid_2d1VrCvdzbY  Iteration_1     10.10.0.11     48645   
3  1.517452e+09  Vid_2d1VrCvdzbY  Iteratio